# Experimental Bead Echo Analysis

This notebook is a compact, top-to-bottom version of the exploratory bead analysis. It uses the tiny `.mat` sample shipped in this repository, so it can be run without the full Eva tracking dataset.

For full-scale training over all echoes, use `experimental_bead_analysis/echo_analysis_pipeline.py`. This notebook is mainly for inspecting the sample data, regenerating lightweight diagnostic plots, and viewing the selected result figures kept in git.


## 1. Setup

The repository is expected to sit next to a local `DCIts/` clone:

```text
workspace/
|-- DCIts/
`-- Interpretable-Deep-Learning-Time-Series/
```


In [ ]:
from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display


def find_experiment_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "support_utils" / "src" / "util_echo.py").exists():
            return candidate
        nested = candidate / "experimental_bead_analysis"
        if (nested / "support_utils" / "src" / "util_echo.py").exists():
            return nested
    raise FileNotFoundError("Could not find experimental_bead_analysis/support_utils.")


EXPERIMENT_ROOT = find_experiment_root(Path.cwd())
REPO_ROOT = EXPERIMENT_ROOT.parent
SUPPORT_ROOT = EXPERIMENT_ROOT / "support_utils"

DCITS_ROOT_CANDIDATES = [
    REPO_ROOT.parent / "DCIts",
    Path.cwd() / "DCIts",
    Path.cwd().parent / "DCIts",
]
DCITS_ROOT = next(
    (path for path in DCITS_ROOT_CANDIDATES if (path / "src" / "dcits.py").exists()),
    None,
)
if DCITS_ROOT is None:
    raise FileNotFoundError(
        "Could not find a sibling DCIts clone. Expected workspace/DCIts next to this repository."
    )

for path in [SUPPORT_ROOT, DCITS_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from src.util_echo import (
    discover_echo_folders,
    load_particles_full_frame,
    make_echo_result_dirs,
    plot_acf_for_clusters,
    plot_all_clusters,
    plot_cluster_time_series,
    plot_individual_clusters,
    save_cluster_members,
    select_clusters_constrained_kmeans,
    build_training_variants,
    train_clusters_for_echo,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    device = torch.device("cuda:0")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("DCITS_ROOT:", DCITS_ROOT)

## 2. Configuration

The default `DATA_ROOT` is the tracked sample-data folder. To reproduce the original full experiment, set `DATA_ROOT` to the complete Eva tracking data folder and run the pipeline script.


In [ ]:
DATA_ROOT = EXPERIMENT_ROOT / "sample_data"
RESULTS_ROOT = EXPERIMENT_ROOT / "artifacts" / "notebook_sample_run"
SELECTED_RESULTS_ROOT = EXPERIMENT_ROOT / "selected_results"

N_SERIES_TO_MODEL = 10
WINDOW_SIZE = 5
ORDER = [1, 1]
TEMPERATURE = 1.0
FIGURE_DPI = 180

# Notebook training is optional because even the tiny sample may take time.
# The pipeline is the preferred way to regenerate full DCIts training outputs.
RUN_DCI_TRAINING = False
QUICK_N_RUNS = 1
QUICK_EPOCHS = 5
QUICK_VARIANTS = ["dxy"]

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

## 3. Discover Available Sample Echoes


In [ ]:
echo_jobs = discover_echo_folders(DATA_ROOT)
jobs_table = pd.DataFrame(echo_jobs)
display(jobs_table[["amplitude_result_name", "echo_name", "echo_dir"]])

# 0 = gamma_1perc sample, 1 = gamma_60perc sample.
SELECTED_JOB_INDEX = 1
job = echo_jobs[SELECTED_JOB_INDEX]

print("Selected:", job["amplitude_result_name"], job["echo_name"])
print("Input folder:", job["echo_dir"])

## 4. Load Particles And Reconstruct The Compact Cluster

The sample folders already contain only one compact 10-bead cluster. Running the constrained clustering step again should therefore produce one cluster with all 10 full-frame particles.


In [ ]:
frames, particle_ids, x_full, y_full = load_particles_full_frame(job["echo_dir"])

clusters = select_clusters_constrained_kmeans(
    particle_ids=particle_ids,
    x_matrix=x_full,
    y_matrix=y_full,
    n_particles=N_SERIES_TO_MODEL,
    frame_index=0,
    random_state=seed,
)

result_dirs = make_echo_result_dirs(
    RESULTS_ROOT,
    job["amplitude_result_name"],
    job["echo_name"],
)

metadata = {
    "amplitude": job["amplitude_result_name"],
    "echo": job["echo_name"],
    "input_dir": str(job["echo_dir"]),
}
save_cluster_members(clusters, result_dirs["root"] / "cluster_members.csv", metadata=metadata)

print("full-frame particles:", len(particle_ids))
print("frames:", len(frames))
print("clusters:", len(clusters))
for idx, cluster in enumerate(clusters):
    print(f"cluster {idx}: size={cluster['cluster_size']}, center={cluster['center_particle_id']}, ids={list(cluster['selected_particle_ids'])}")

## 5. Lightweight Diagnostic Plots

These cells regenerate only simple diagnostic figures. They do not train DCIts unless `RUN_DCI_TRAINING = True` later.


In [ ]:
plot_all_clusters(
    x_full=x_full,
    y_full=y_full,
    clusters=clusters,
    save_path=result_dirs["root"] / "clusters.png",
    figure_dpi=FIGURE_DPI,
)

plot_individual_clusters(
    x_full=x_full,
    y_full=y_full,
    particle_ids=particle_ids,
    clusters=clusters,
    output_dir=result_dirs["individual_clusters"],
    figure_dpi=FIGURE_DPI,
)

plot_cluster_time_series(
    frames=frames,
    clusters=clusters,
    output_dir=result_dirs["individual_cluster_time_series"],
    figure_dpi=FIGURE_DPI,
    event_percentile=99.6,
)

plot_acf_for_clusters(
    clusters=clusters,
    frames=frames,
    output_dir=result_dirs["acf"],
    figure_dpi=FIGURE_DPI,
    lags=10,
    include_shuffled=True,
)

print("Generated diagnostic plots in:", result_dirs["root"])

In [ ]:
def show_image(path, width=900):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print("Missing:", path)

show_image(result_dirs["root"] / "clusters.png", width=600)
show_image(result_dirs["individual_clusters"] / "cluster_0.png", width=800)
show_image(result_dirs["individual_cluster_time_series"] / "cluster_series_0.png", width=900)

## 6. Build DCIts Input Variants

The original analysis compared three representations:

- `raw_xy`: raw x/y positions;
- `rel_xy`: positions relative to the first frame;
- `dxy`: frame-to-frame displacements.


In [ ]:
cluster = clusters[0]
variants = build_training_variants(
    cluster["x_selected"],
    cluster["y_selected"],
    frames,
    include_shuffled=True,
)

for variant_name, (time_series, frames_used) in variants.items():
    print(f"{variant_name:15s} time_series={tuple(time_series.shape)} frames={len(frames_used)}")

# Event table used for event-conditioned inspection.
y_selected = cluster["y_selected"]
selected_ids = np.asarray(cluster["selected_particle_ids"])
dy = np.diff(y_selected, axis=0)
abs_dy = np.abs(dy)
event_percentile = 99.6
threshold = np.percentile(abs_dy, event_percentile)
event_time_idx, event_particle_idx = np.where(abs_dy >= threshold)

event_table = pd.DataFrame({
    "bead_id": selected_ids[event_particle_idx],
    "frame_before": frames[event_time_idx],
    "frame_after": frames[event_time_idx + 1],
    "dy": dy[event_time_idx, event_particle_idx],
    "abs_dy": abs_dy[event_time_idx, event_particle_idx],
}).sort_values("abs_dy", ascending=False)

display(event_table.head(12))

## 7. Statistical Exploratory Tests

These tests are exploratory diagnostics for one representative bead pair. They are not used as DCIts inputs; they just document the classical time-series checks that were tried during the bead analysis.


In [ ]:
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings

series_a = pd.Series(cluster["y_selected"][:, 0], name=f"y bead {selected_ids[0]}")
diff_a = pd.Series(np.diff(cluster["y_selected"][:, 0]), name=f"dy bead {selected_ids[0]}")


def adf_summary_row(series, label):
    stat, pvalue, used_lag, nobs, *_ = adfuller(series.dropna().values)
    return {
        "series": label,
        "ADF statistic": stat,
        "p-value": pvalue,
        "used lags": used_lag,
        "n obs": nobs,
    }

adf_summary = pd.DataFrame([
    adf_summary_row(series_a, series_a.name),
    adf_summary_row(diff_a, diff_a.name),
])
display(adf_summary)

ljung_box = acorr_ljungbox(diff_a, lags=[5, 10, 20], return_df=True)
display(ljung_box)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(diff_a, lags=40, ax=axes[0])
plot_pacf(diff_a, lags=40, method="ywm", ax=axes[1])
axes[0].set_title(f"ACF: {diff_a.name}")
axes[1].set_title(f"PACF: {diff_a.name}")
plt.tight_layout()
plt.show()

if cluster["y_selected"].shape[1] >= 2:
    diff_b = pd.Series(np.diff(cluster["y_selected"][:, 1]), name=f"dy bead {selected_ids[1]}")
    granger_data = pd.DataFrame({"target": diff_a, "source": diff_b}).dropna()

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            granger_result = grangercausalitytests(
                granger_data[["target", "source"]],
                maxlag=3,
                verbose=False,
            )

        granger_summary = pd.DataFrame([
            {
                "lag": lag,
                "ssr F-test p-value": result[0]["ssr_ftest"][1],
                "source": diff_b.name,
                "target": diff_a.name,
            }
            for lag, result in granger_result.items()
        ])
        display(granger_summary)
    except Exception as exc:
        print("Granger test skipped:", exc)


## 8. Inline Supervised Training Experiments

The original notebook also tested whether DCIts can predict supervised targets from bead-motion inputs: movement magnitude `abs(delta raw_xy)` and a binary event signal `E(t)`. These cells keep that experiment available, but training is off by default because the saved figures in `selected_results/` are already included.


In [ ]:
from src.utils_dipl import collect_multiple_runs_supervised
import torch.nn as nn

RUN_INLINE_SUPERVISED = False
SUPERVISED_INPUT_VARIANT = "dxy"  # one of: raw_xy, rel_xy, dxy
SUPERVISED_N_RUNS = 1
SUPERVISED_EPOCHS = 5
EVENT_QUANTILE = 0.95

raw_xy = variants["raw_xy"][0].float()
rel_xy = variants["rel_xy"][0].float()
dxy = variants["dxy"][0].float()

supervised_inputs = {
    "raw_xy": raw_xy[:, :-1],
    "rel_xy": rel_xy[:, :-1],
    "dxy": dxy,
}

abs_delta_raw_xy = torch.abs(raw_xy[:, 1:] - raw_xy[:, :-1])
event_threshold = torch.quantile(abs_delta_raw_xy, EVENT_QUANTILE, dim=1, keepdim=True)
event_target = (abs_delta_raw_xy >= event_threshold).float()
supervised_input = supervised_inputs[SUPERVISED_INPUT_VARIANT]

print("input variant:", SUPERVISED_INPUT_VARIANT, tuple(supervised_input.shape))
print("abs(delta raw_xy) target:", tuple(abs_delta_raw_xy.shape))
print("E(t) target:", tuple(event_target.shape), "positive fraction:", float(event_target.mean()))


In [ ]:
def summarize_supervised_regression(results, label):
    run_keys = [key for key in results if key.startswith("run_")]
    losses = np.array([results[key]["test_loss"] for key in run_keys])
    first_test = results[run_keys[0]]["split_results"]["test"]
    pred = first_test["predictions"].detach().cpu()
    target = first_test["targets"].detach().cpu()
    mae = torch.mean(torch.abs(pred - target)).item()
    print(f"{label} test MSE: {losses.mean():.6g} +/- {losses.std():.6g}")
    print(f"{label} run_1 test MAE: {mae:.6g}")


def summarize_supervised_event(results, label):
    run_keys = [key for key in results if key.startswith("run_")]
    losses = np.array([results[key]["test_loss"] for key in run_keys])
    first_test = results[run_keys[0]]["split_results"]["test"]
    logits = first_test["predictions"].detach().cpu()
    target = first_test["targets"].detach().cpu()
    prob = torch.sigmoid(logits)
    pred = (prob >= 0.5).float()
    accuracy = (pred == target).float().mean().item()
    print(f"{label} test BCE: {losses.mean():.6g} +/- {losses.std():.6g}")
    print(f"{label} run_1 accuracy: {accuracy:.4f}")
    print(f"{label} predicted positive fraction: {float(pred.mean()):.4f}")


if RUN_INLINE_SUPERVISED:
    base_supervised_config = {
        "verbose": False,
        "device": device,
        "seed": seed,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "scheduler_patience": 3,
        "early_stopping_modifier": 2,
        "epochs": SUPERVISED_EPOCHS,
        "min_epochs": 1,
    }

    abs_delta_config = dict(base_supervised_config)
    abs_delta_config["criterion"] = nn.MSELoss()
    abs_delta_results = collect_multiple_runs_supervised(
        n_runs=SUPERVISED_N_RUNS,
        input_series=supervised_input,
        target_series=abs_delta_raw_xy,
        window_size=WINDOW_SIZE,
        temperature=TEMPERATURE,
        order=ORDER,
        config=abs_delta_config,
        seed=seed,
        verbose=True,
    )
    summarize_supervised_regression(abs_delta_results, "abs(delta raw_xy)")

    event_config = dict(base_supervised_config)
    event_config["criterion"] = nn.BCEWithLogitsLoss()
    event_results = collect_multiple_runs_supervised(
        n_runs=SUPERVISED_N_RUNS,
        input_series=supervised_input,
        target_series=event_target,
        window_size=WINDOW_SIZE,
        temperature=TEMPERATURE,
        order=ORDER,
        config=event_config,
        seed=seed + 100,
        verbose=True,
    )
    summarize_supervised_event(event_results, "E(t)")
else:
    print("Inline supervised training is prepared but skipped.")
    print("Set RUN_INLINE_SUPERVISED = True to train abs(delta raw_xy) and E(t) models here.")


## 9. Optional DCIts Training

Leave `RUN_DCI_TRAINING = False` for a fast notebook smoke run. Set it to `True` only when you want to regenerate small DCIts outputs from the sample data. For the full thesis outputs, use the pipeline script instead.


In [ ]:
if RUN_DCI_TRAINING:
    training_metadata = {
        "amplitude": job["amplitude_result_name"],
        "echo": job["echo_name"],
        "input_dir": str(job["echo_dir"]),
        "n_particles_full_frame": len(particle_ids),
        "n_clusters": len(clusters),
        "n_series_to_model": N_SERIES_TO_MODEL,
        "window_size": WINDOW_SIZE,
        "n_runs": QUICK_N_RUNS,
        "epochs": QUICK_EPOCHS,
    }

    training_summary = train_clusters_for_echo(
        clusters=clusters,
        frames=frames,
        result_dirs=result_dirs,
        device=device,
        seed=seed,
        n_runs=QUICK_N_RUNS,
        window_size=WINDOW_SIZE,
        temperature=TEMPERATURE,
        order=ORDER,
        epochs=QUICK_EPOCHS,
        figure_dpi=FIGURE_DPI,
        cluster_limit=1,
        variant_names=QUICK_VARIANTS,
        include_shuffled=False,
        metadata=training_metadata,
    )
    display(training_summary.head())
else:
    print("Skipping DCIts training in the notebook.")
    print("For the full run, use experimental_bead_analysis/echo_analysis_pipeline.py.")

## 10. View Selected Results Kept In Git

These figures were generated by earlier full runs and selected for GitHub. They are separate from the notebook-generated `artifacts/` folder.


In [ ]:
case_key = f"{job['amplitude_result_name']}_{job['echo_name']}"
case_results = SELECTED_RESULTS_ROOT / case_key
print("Selected result folder:", case_results)

representative_images = [
    case_results / "clusters.png",
    case_results / "individual_clusters" / "cluster_0.png",
    case_results / "individual_cluster_time_series" / "cluster_series_0.png",
    case_results / "heatmaps" / "cluster_0" / "alpha_heatmap_test_dxy.png",
    case_results / "heatmaps" / "cluster_0" / "alpha_heatmap_test_dxy_shuffled.png",
    case_results / "self_alpha_lag1" / "cluster_0" / "self_alpha_lag1_batch_0_0.png",
    case_results / "event_interpretability_plots" / "cluster_0" / "selected_event_time_series.png",
    case_results / "event_interpretability_plots" / "cluster_0" / "average_heatmaps" / "alpha.png",
]

for image_path in representative_images:
    if image_path.exists():
        print(image_path.relative_to(EXPERIMENT_ROOT))
        show_image(image_path, width=850)

## 11. Saved Notebook-End Supervised Results

The original exploratory notebook also tested two supervised variants at the end:

- `supervised_abs_delta_y`: predict movement magnitude, `raw_xy / rel_xy / dxy -> |delta raw_xy|`;
- `supervised_event_Et`: predict a binary event indicator `E(t)`.

The tracked figures below are selected outputs from those exploratory cells.


In [ ]:
extra_result_dirs = [
    SELECTED_RESULTS_ROOT / "gamma_60perc_echo1" / "supervised_abs_delta_y",
    SELECTED_RESULTS_ROOT / "gamma_60perc_echo1" / "supervised_event_Et",
]

for folder in extra_result_dirs:
    print("\n", folder.relative_to(EXPERIMENT_ROOT))
    if not folder.exists():
        print("Missing")
        continue
    for image_path in sorted(folder.rglob("*.png"))[:6]:
        print(image_path.relative_to(EXPERIMENT_ROOT))
        show_image(image_path, width=850)

## 12. Full Pipeline Command

From the repository root, the smallest no-training smoke command is:

```powershell
python experimental_bead_analysis\echo_analysis_pipeline.py --echo-limit 2 --cluster-limit 1 --skip-training --skip-acf-pacf --skip-granger --skip-adf
```

A training run on the sample data is:

```powershell
python experimental_bead_analysis\echo_analysis_pipeline.py --echo-limit 2 --cluster-limit 1 --epochs 5 --n-runs 1 --no-shuffled-controls
```

For the full Eva data folder, add `--data-root "path\to\complete\data"` and use the commands in `experimental_bead_analysis/run_commands.txt`.
